In [11]:
from read_training_data import get_tension_data, get_raw_data, get_lengths
import pandas as pd
import numpy as np


In [12]:
raw_data_path = "training-data/raw/APA_5947a8b0-5152-11ed-84c6-9170f34dd096b_apa4_training"
processed_data_path = "training-data/processed/5947a8b0-5152-11ed-84c6-9170f34dd096b_apa4_training.json"

In [13]:
proc_scans = get_tension_data(processed_data_path, ["U", "X"])


In [14]:
keys_to_remove = {"scanEndMode", "measuredBy", "stage", "apaUuid", "type"}
raw_scans = get_raw_data(raw_data_path, keys_to_remove)

In [15]:
proc_df = pd.DataFrame(proc_scans)
raw_df = pd.DataFrame(raw_scans)
# raw_df.head()

In [16]:
proc_df.head()

,uid,stage,tension
0,UA34,Winding,5.573973
1,UA36,Winding,5.408697
2,UA38,Winding,8.020985
3,UA40,Winding,5.693777
4,UA41,Winding,5.326113


In [17]:
comb_df = pd.merge(raw_df, proc_df, on="uid", how="outer")
comb_df.head(3)

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension
0,UA1002,1002,0,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3870.976801, 3870.854905, 3875.409452, 3883.0...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.1644
1,UA1004,1004,1,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[1328.26138, 1327.855029, 1329.099351, 1332.22...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.331951
2,UA1006,1006,2,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3779.794773, 3780.85742, 3783.779545, 3791.47...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.381051


In [ ]:
final_comb_df = comb_df.dropna(subset=['tension'])
numeric_tension = pd.to_numeric(final_comb_df['tension'], errors='coerce') # removes tensions that aren't found
final_comb_df['tension'] = numeric_tension
final_comb_df.shape

(698, 11)

In [19]:
u_side_lengths = get_lengths("training-data/tensions/U_Layer_Tension_Template.xlsx", "U")

In [20]:
u_side_lengths.columns

Index(['length', 'uid'], dtype='str')

In [30]:
u_side_lengths["length"].values

array([69.86160506, 79.70761336, 89.55523435, ..., 70.95958991,
       61.1647848 , 51.39186122], shape=(2282,))

In [21]:
final_df = pd.merge(final_comb_df, u_side_lengths, on="uid", how="inner")


In [32]:
final_df.shape

(618, 12)

In [41]:
print(type(final_df['length'].values))
print(type(final_df['tension'].values))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [53]:
# Create a mask that checks if each item is a string
string_mask = final_df['tension'].apply(lambda x: isinstance(x, str))

# Display the rows that match the mask
numeric_tension = pd.to_numeric(final_df['tension'], errors='coerce') # converts tension not found to NaN
final_df['tension'] = numeric_tension
final_df = final_df.dropna(subset=['tension']) # drops rows with NaN tension values
final_df.shape

(534, 12)

In [54]:
print(final_df['tension'].values[10])
print(final_df['length'].values[10])

7.093048586787
1272.5719246470906


In [57]:
tension = final_df['tension'].values
lengths = final_df['length'].values / 1000 # in meters

rho = 1.52e-4

res_freq = np.sqrt(tension / rho) * (1 / (2*lengths))

res_freq

final_df['res_freq'] = res_freq

In [58]:
final_df.head(3)

,uid,segment_id,channel_index,freq,ampl,layer,headboardNumber,side,parent_folder,stage,tension,length,res_freq
0,UA1002,1002,0,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3870.976801, 3870.854905, 3875.409452, 3883.0...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.164400,1489.313716,67.609454
1,UA1004,1004,1,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[1328.26138, 1327.855029, 1329.099351, 1332.22...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.331951,1469.609881,69.440834
2,UA1006,1006,2,"[50.0, 50.125, 50.25, 50.375, 50.5, 50.625, 50...","[3779.794773, 3780.85742, 3783.779545, 3791.47...",U,5,A,training-data/raw/APA_5947a8b0-5152-11ed-84c6-...,Winding,6.381051,1449.906053,70.656882
